# Native MXFP8 Type in Allo

This tutorial shows how to write Allo kernels that use the **compiler-native** `mxfp8` type
and call MXFP8 **intrinsics** directly.

Compared to [Tutorial 03](tutorial_03_mxfp8.ipynb), which uses ready-made library kernels
(`mxfp8.mxfp8_block_add`), here you:

1. Annotate payloads as `mxfp8[BS]` instead of raw `uint8[BS]`
2. Call `allo.mxfp8_ops.*` intrinsics from your own kernel
3. Inspect the resulting MLIR (`!allo.Mxfp8<32>`, `allo.block_add_mxfp8`)
4. Run LLVM simulation and emit Vivado HLS from the same schedule

**Prerequisites**: `conda activate allo` and a built Allo install (`pip install -v -e .`).

## MXFP8 layout (unchanged)

Each block has 32 elements:

- **Scale**: one `uint8` E8M0 byte (shared block exponent, power-of-two only)
- **Payload**: `mxfp8[32]` — 32 × E4M3 bytes stored as a native memref type

```python
scale: uint8          # E8M0
data:  mxfp8[32]     # native payload (lowers to uint8 in HLS)
```

Use `ref_*` helpers from `allo.library.mxfp8` only for **test data** and golden verification.
Your kernel should call compiler intrinsics in `allo.mxfp8_ops`.

In [10]:
import numpy as np
import allo
from pathlib import Path
from allo import mxfp8_ops
from allo.ir.types import mxfp8, uint8, float32, int32
from IPython.display import Code, display
from allo.library.mxfp8 import (
    MXFP8_BLOCK_SIZE,
    ref_encode_block,
    ref_decode_block,
    ref_block_add,
)

## Step 1 — Prepare test inputs

Encode two random float32 vectors with the Python reference encoder. This gives us
MXFP8 blocks to feed into our native kernel and a golden `a + b` result.

In [11]:
bs = MXFP8_BLOCK_SIZE  # 32
np.random.seed(0)

a = np.random.randn(bs).astype(np.float32)
b = np.random.randn(bs).astype(np.float32)

scale_a, data_a = ref_encode_block(a)
scale_b, data_b = ref_encode_block(b)
golden = a + b

print("scale_a:", scale_a, " scale_b:", scale_b)
print("expected a+b (first 4):", golden[:4])

scale_a: 129  scale_b: 128
expected a+b (first 4): [ 0.87626666 -1.5806392   0.6308259   2.397242  ]


## Step 2 — Scalar intrinsics

Start with a tiny kernel that decodes a single E4M3 byte. Functions in `mxfp8_ops`
are **compiler intrinsics** — they raise at Python runtime but lower to `allo.decode_e4m3`
MLIR ops during compilation.

In [12]:
def decode_one(u8: uint8) -> float32:
    return mxfp8_ops.decode_e4m3(u8)


scalar_mod = allo.customize(decode_one).build()
print("decode_e4m3(42) =", scalar_mod(42))

decode_e4m3(42) = 0.3125


## Step 3 — Define a kernel with `mxfp8[BS]` payloads

The block-add kernel takes separate scale bytes and `mxfp8` payload memrefs.
One intrinsic performs decode → add → re-encode inside the compiler.

In [13]:
def native_block_add[BS: int32](
    scale_a: uint8,
    data_a: mxfp8[BS],
    scale_b: uint8,
    data_b: mxfp8[BS],
    scale_out: uint8[1],
    data_out: mxfp8[BS],
):
    mxfp8_ops.block_add_mxfp8(
        scale_a, data_a, scale_b, data_b, scale_out, data_out
    )

## Step 4 — Customize and inspect MLIR

After `allo.customize`, the payload memrefs appear as `!allo.Mxfp8<BS>` and the
block operation becomes a single `allo.block_add_mxfp8` op (not a Python loop).

In [14]:
s = allo.customize(native_block_add, instantiate=[bs])

ir_lines = str(s.module).splitlines()
for line in ir_lines:
    if "Mxfp8" in line or "block_add_mxfp8" in line:
        print(line)

  func.func @native_block_add(%arg0: i8, %arg1: memref<32x!allo.Mxfp8<32>>, %arg2: i8, %arg3: memref<32x!allo.Mxfp8<32>>, %arg4: memref<1xi8>, %arg5: memref<32x!allo.Mxfp8<32>>) attributes {itypes = "u_u_u_", otypes = ""} {
    allo.block_add_mxfp8(%arg0, %arg1, %arg2, %arg3, %arg4, %arg5) : i8, memref<32x!allo.Mxfp8<32>>, i8, memref<32x!allo.Mxfp8<32>>, memref<1xi8>, memref<32x!allo.Mxfp8<32>>


## Step 5 — LLVM simulation

At runtime, NumPy still passes payload arrays as `uint8` buffers; the compiler treats
them as `mxfp8` memrefs. Decode the output with `ref_decode_block` to compare against
float32 golden values.

In [15]:
mod = s.build()

scale_out = np.zeros(1, dtype=np.uint8)
data_out = np.zeros(bs, dtype=np.uint8)
mod(int(scale_a), data_a, int(scale_b), data_b, scale_out, data_out)

result = ref_decode_block(int(scale_out[0]), data_out)
print("LLVM result (first 4):", result[:4])
print("Expected a+b (first 4):", golden[:4])
np.testing.assert_allclose(result, golden, atol=0.5)
print("LLVM simulation passed.")

LLVM result (first 4): [ 0.875  -1.625   0.6875  2.5   ]
Expected a+b (first 4): [ 0.87626666 -1.5806392   0.6308259   2.397242  ]
LLVM simulation passed.


## Step 6 — Block decode with a native intrinsic

You can also decode an entire block to float32 inside the kernel:

In [16]:
def native_decode_block[BS: int32](
    scale: uint8, data: mxfp8[BS], out: float32[BS]
):
    mxfp8_ops.decode_mxfp8_block(scale, data, out)


decode_mod = allo.customize(native_decode_block, instantiate=[bs]).build()
decoded_a = np.zeros(bs, dtype=np.float32)
decode_mod(int(scale_a), data_a, decoded_a)
np.testing.assert_allclose(decoded_a, a, atol=0.5)
print("Block decode passed. First 4:", decoded_a[:4])

Block decode passed. First 4: [1.75    0.40625 1.      2.25   ]


## Step 7 — Vivado HLS codegen

HLS emission uses shared device helpers (`allo_block_add_mxfp8`, `allo_decode_e4m3`, …)
instead of inlining decode/encode logic into every user kernel.

The cell below saves the full generated C++ next to this notebook (path printed after save).
Open that file in your editor if the notebook view is truncated.

In [17]:
hls_mod = s.build(target="vhls")
hls_code = hls_mod.hls_code

assert "allo_block_add_mxfp8" in hls_code
assert "allo_decode_e4m3" in hls_code

hls_path = Path("native_block_add.hls.cpp")
hls_path.write_text(hls_code)

print(f"Generated HLS: {len(hls_code.splitlines())} lines")
print(f"Full code saved to: {hls_path.resolve()}")
print("Tip: open that file in the editor if the notebook view is truncated.\n")

display(Code(hls_code, language="cpp"))

Generated HLS: 140 lines
Full code saved to: /home/rbdus0715/allo/tutorials/native_block_add.hls.cpp
Tip: open that file in the editor if the notebook view is truncated.



//===------------------------------------------------------------*- C++ -*-===//
//
// Automatically generated file for High-level Synthesis (HLS).
//
//===----------------------------------------------------------------------===//
#include <algorithm>
#include <ap_axi_sdata.h>
#include <ap_fixed.h>
#include <ap_int.h>
#include <hls_math.h>
#include <hls_stream.h>
#include <hls_vector.h>
#include <math.h>
#include <stdint.h>
using namespace std;

static const int ALLO_E4M3_BIAS = 7;
static const int ALLO_E8M0_BIAS = 127;

static inline float allo_pow2_int(int exp) { return ldexp(1.0f, exp); }

static inline float allo_decode_e4m3(uint8_t u8) {
  int sign = (u8 >> 7) & 1;
  int exp = (u8 >> 3) & 15;
  int mant = u8 & 7;
  float val = 0.0f;
  if (exp == 15 && mant == 7) {
    val = 0.0f;
  } else if (exp == 0) {
    val = float(mant) / 8.0f * allo_pow2_int(1 - ALLO_E4M3_BIAS);
  } else {
    val = (1.0f + float(mant) / 8.0f) * allo_pow2_int(exp - ALLO_E4M3_BIAS);
  }
  return sign ? -val : val;
}

static inline uint8_t allo_encode_e4m3(float f) {
  if (f == 0.0f)
    return 0;
  int sign = (f < 0.0f) ? 1 : 0;
  float abs_f = sign ? -f : f;
  int unbiased_exp = -20;
  for (int e = -20; e < 16; ++e) {
    if (abs_f >= allo_pow2_int(e))
      unbiased_exp = e;
  }
  int exp_field = unbiased_exp + ALLO_E4M3_BIAS;
  int mant = 0;
  if (exp_field <= 0) {
    exp_field = 0;
    mant = int(abs_f / allo_pow2_int(1 - ALLO_E4M3_BIAS) * 8.0f + 0.5f);
  } else {
    mant = int((abs_f / allo_pow2_int(unbiased_exp) - 1.0f) * 8.0f + 0.5f);
    if (mant == 8) {
      mant = 0;
      exp_field += 1;
    }
  }
  if (exp_field >= 15) {
    exp_field = 14;
    mant = 7;
  }
  return uint8_t((sign << 7) | (exp_field << 3) | mant);
}

static inline float allo_decode_e8m0(uint8_t u8) {
  if (u8 == 0 || u8 == 255)
    return 0.0f;
  return allo_pow2_int(int(u8) - ALLO_E8M0_BIAS);
}

static inline uint8_t allo_encode_e8m0(float scale) {
  if (scale <= 0.0f)
    return 0;
  int result = 254;
  for (int e = 1; e < 255; ++e) {
    if (result == 254 && allo_pow2_int(e - ALLO_E8M0_BIAS) >= scale)
      result = e;
  }
  return uint8_t(result);
}

static inline void allo_decode_mxfp8_block(int bs, uint8_t scale, uint8_t *data,
                                           float *out) {
  float s = allo_decode_e8m0(scale);
  for (int i = 0; i < bs; ++i)
    out[i] = allo_decode_e4m3(data[i]) * s;
}

static inline void allo_encode_mxfp8_block(int bs, float *data, uint8_t *scale_out,
                                           uint8_t *data_out) {
  float max_val = 0.0f;
  for (int i = 0; i < bs; ++i) {
    float av = data[i] < 0.0f ? -data[i] : data[i];
    if (av > max_val)
      max_val = av;
  }
  scale_out[0] = allo_encode_e8m0(max_val);
  float s = allo_decode_e8m0(scale_out[0]);
  for (int i = 0; i < bs; ++i) {
    float scaled = (s != 0.0f) ? data[i] / s : data[i];
    data_out[i] = allo_encode_e4m3(scaled);
  }
}

static inline void allo_block_add_mxfp8(int bs, uint8_t scale_a, uint8_t *data_a,
                                        uint8_t scale_b, uint8_t *data_b,
                                        uint8_t *scale_out, uint8_t *data_out) {
  float sa = allo_decode_e8m0(scale_a);
  float sb = allo_decode_e8m0(scale_b);
  float buf[32];
  for (int i = 0; i < bs; ++i)
    buf[i] = allo_decode_e4m3(data_a[i]) * sa + allo_decode_e4m3(data_b[i]) * sb;
  allo_encode_mxfp8_block(bs, buf, scale_out, data_out);
}

static inline void allo_block_matmul_mxfp8(int bs, uint8_t scale_a, uint8_t *data_a,
                                           uint8_t scale_b, uint8_t *data_b,
                                           uint8_t *scale_out, uint8_t *data_out) {
  float sa = allo_decode_e8m0(scale_a);
  float sb = allo_decode_e8m0(scale_b);
  float acc = 0.0f;
  for (int i = 0; i < bs; ++i)
    acc += allo_decode_e4m3(data_a[i]) * sa * allo_decode_e4m3(data_b[i]) * sb;
  float out[1] = {acc};
  allo_encode_mxfp8_block(1, out, scale_out, data_out);
}
/// This is top fu

## Step 8 — Block dot product (optional)

For a 1×32 · 32×1 inner product encoded as a single MXFP8 output element, use
`mxfp8_ops.block_matmul_mxfp8`:

In [18]:
def native_block_matmul[BS: int32](
    scale_a: uint8,
    data_a: mxfp8[BS],
    scale_b: uint8,
    data_b: mxfp8[BS],
    scale_out: uint8[1],
    data_out: mxfp8[BS],
):
    mxfp8_ops.block_matmul_mxfp8(
        scale_a, data_a, scale_b, data_b, scale_out, data_out
    )


matmul_mod = allo.customize(native_block_matmul, instantiate=[bs]).build()
so = np.zeros(1, dtype=np.uint8)
dout = np.zeros(bs, dtype=np.uint8)
matmul_mod(int(scale_a), data_a, int(scale_b), data_b, so, dout)

dot = float(np.sum(a * b))
approx = ref_decode_block(int(so[0]), dout)[0]
print(f"dot product ≈ {approx:.4f}  (float32 golden {dot:.4f})")
assert abs(approx - dot) < max(0.5, abs(dot) * 0.5)

dot product ≈ -5.5000  (float32 golden -5.5462)


## Summary

| Layer | What you write | What the compiler sees |
|-------|----------------|------------------------|
| Payload type | `mxfp8[32]` | `memref<32x!allo.Mxfp8<32>>` |
| Block add | `mxfp8_ops.block_add_mxfp8(...)` | `allo.block_add_mxfp8` |
| LLVM sim | `s.build()` | `mxfp8-to-primitive` pass expands to float ops |
| VHLS | `s.build(target="vhls")` | Shared `allo_*` C++ helpers |

**Current limitations (Phase 1):**

- No expression-level `data_a + data_b` overloading — use explicit intrinsics
- Scale stays a separate `uint8`; only the payload uses `mxfp8[BS]`
- Library kernels in `allo.library.mxfp8` are thin wrappers over the same intrinsics

See also: `examples/mxfp8_block_add.py` (library path) and `docs/source/dive/frontend_syntax.rst` (MXFP8 section).